In [ ]:
import sys
!{sys.executable} -m pip install krylov

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import krylov
import matplotlib.pyplot as plt
import sys  
from scipy import linalg as scla
from tqdm import tqdm
mypath = '/home/bbb/Galerkin-Differencing/general_solve'
sys.path.insert(1, mypath)
from general_solve.stokes import StokesFlow

In [ ]:
def matvis(m,name=None):
    vism = m.copy()
    vism[vism == 0] = np.nan
    plt.matshow(vism)
    plt.show()

In [ ]:
rtypes = ['uniform','stripe','square']
rnames = {'stripe':['vertfinecenter',
					'vertcoarsecenter',
					'horzfinecenter',
					'horzcoarsecenter'],
		  'square':['finecenter','coarsecenter']}

In [ ]:
# setup u,v,p,f
mu = 1
rho = 1

u = lambda x,y: -2*np.cos(2*np.pi*x)*np.sin(2*np.pi*y)
ulap = lambda x,y: -8*np.pi**2*u(x,y)

v = lambda x,y: 2*np.sin(2*np.pi*x)*np.cos(2*np.pi*y)
vlap = lambda x,y: -8*np.pi**2*v(x,y)

p = lambda x,y: -np.cos(4*np.pi*x)+np.cos(4*np.pi*y)

f_0 = lambda x,y: 4*np.pi*(2*np.pi*u(x,y)-np.sin(4*np.pi*x))
f_1 = lambda x,y: 4*np.pi*(2*np.pi*v(x,y)+np.sin(4*np.pi*y))

In [ ]:
N = 8
mystokes = StokesFlow(N,ord=1,vars=[u,v,p],rtype='uniform',rname='vertcoarsecenter')

In [ ]:
vars = [mystokes.velocity.u,mystokes.velocity.v,mystokes.pressure]

for var in vars:
	for p in var.mesh.patches:
		p.vis()#rtype='uniform')
		p.vis_interface_eval_points(rtype='stripe')

In [ ]:
dv,lp = mystokes.check_schur_null()

In [ ]:
matvis(lp)

In [ ]:
mystokes.solve([f_0,f_1])

In [ ]:
sys = mystokes.sys.todense()
ns = scla.null_space(sys)

In [ ]:
ns.shape

In [ ]:
sys.shape

In [ ]:
mystokes.rhs.shape

In [ ]:
plt.plot(ns)

In [ ]:
for vec in ns.T:
	print(mystokes.rhs @ vec)
	full_ns = mystokes.C.dot(vec)
	ns_list = mystokes._split_vec(full_ns)
	# print(full_ns.shape,mystokes3.sol_vec.shape)
	mystokes.view_sol(ns_list,err=False)

In [ ]:
mystokes.C_sizes, mystokes.sys.shape

In [ ]:
mystokes.laplace.shape, mystokes.divergence.shape

In [ ]:
mystokes.velocity.spC.shape, mystokes.pressure.constraints.spC.shape

In [ ]:
256*3

In [ ]:
mystokes.A.shape, mystokes.C.shape

In [ ]:
mystokes.pressure.constraints.spC.shape
mystokes.velocity.v.constraints.spC.shape

In [ ]:
schur = mystokes.check_schur_null()

In [ ]:
mystokes.view_sol(mystokes.sol_vecs)

In [ ]:
Ns = np.array([16,32])
errs = np.zeros((len(Ns),4))
for index,N in enumerate(Ns):
	mystokes = StokesFlow(N,ords=[1,1,1],vars=[u,v,p])
	mystokes.solve([f_0,f_1],disp=False)
	errs[index] = [mystokes.velocity.L2_err,mystokes.pressure.L2_err,
				   mystokes.velocity.Linf_err,mystokes.pressure.Linf_err]

In [ ]:
fig = plt.figure(figsize=(20,8))
plt.subplot(121)
plt.loglog(Ns,1/Ns**2,label=r'$h^2$')
plt.loglog(Ns,errs[:,0],label=r'$|\hat{u}|_{L_2}$')
plt.loglog(Ns,errs[:,1],label=r'$|{p}|_{L_2}$')
plt.legend(fontsize=20)
plt.subplot(122)
plt.loglog(Ns,1/Ns**1,label=r'$h$')
plt.loglog(Ns,errs[:,2],label=r'$|\hat{u}|_{L_\infty}$')
plt.loglog(Ns,errs[:,3],label=r'$|p|_{L_\infty}$')
plt.legend(fontsize=20)
plt.suptitle('Convergence for Uniform Stokes',fontsize=20)
plt.savefig('uniform_stokes_convergence.png',dpi=300)
plt.show()

In [ ]:
Ns = np.array([16,32])
errs = np.zeros((len(Ns),4))
for index,N in enumerate(Ns):
	mystokes = StokesFlow(N,ords=[2,1,1],vars=[u,v,p],
					      rtype='stripe',rname='vertcoarsecenter')
	mystokes.solve([f_0,f_1],disp=False)
	errs[index] = [mystokes.velocity.L2_err,mystokes.pressure.L2_err,
				   mystokes.velocity.Linf_err,mystokes.pressure.Linf_err]

In [ ]:
fig = plt.figure(figsize=(20,8))
plt.subplot(121)
plt.loglog(Ns,1/Ns**2,label=r'$h^2$')
plt.loglog(Ns,errs[:,0],label=r'$|\hat{u}|_{L_2}$')
plt.loglog(Ns,errs[:,1],label=r'$|{p}|_{L_2}$')
plt.legend(fontsize=20)
plt.subplot(122)
plt.loglog(Ns,1/Ns**1,label=r'$h$')
plt.loglog(Ns,errs[:,2],label=r'$|\hat{u}|_{L_\infty}$')
plt.loglog(Ns,errs[:,3],label=r'$|p|_{L_\infty}$')
plt.legend(fontsize=20)
plt.suptitle('Convergence for Striped Stokes',fontsize=20)
plt.savefig('stripe_stokes_convergence.png',dpi=300)
plt.show()

In [ ]:
errs[:,0],errs[:,0]/4

In [ ]:
Ns = np.array([16,32])
errs = np.zeros((len(Ns),4))
for index,N in enumerate(Ns):
	mystokes = StokesFlow(N,ords=[1,1,1],vars=[u,v,p],
					      rtype='square',rname='finecenter')
	mystokes.solve([f_0,f_1],disp=False)
	errs[index] = [mystokes.velocity.L2_err,mystokes.pressure.L2_err,
				   mystokes.velocity.Linf_err,mystokes.pressure.Linf_err]

In [ ]:
fig = plt.figure(figsize=(20,8))
plt.subplot(121)
plt.loglog(Ns,1/Ns**2,label=r'$h^3$')
plt.loglog(Ns,errs[:,0],label=r'$|\hat{u}|_{L_2}$')
plt.loglog(Ns,errs[:,1],label=r'$|{p}|_{L_2}$')
plt.legend(fontsize=20)
plt.subplot(122)
plt.loglog(Ns,1/Ns**1,label=r'$h$')
plt.loglog(Ns,errs[:,2],label=r'$|\hat{u}|_{L_\infty}$')
plt.loglog(Ns,errs[:,3],label=r'$|p|_{L_\infty}$')
plt.legend(fontsize=20)
plt.suptitle('Convergence for Striped Stokes',fontsize=20)
plt.savefig('stripe_stokes_convergence.png',dpi=300)
plt.show()

In [ ]:
errs[:,0],errs[:,0]/4